# 🧠 Memory & Threads in Conversational Agentic Systems

By default, every interaction with an agent is an isolated request — it has no idea what
you said a moment ago. **Memory** and **threads** are what turn a stateless graph into a
conversational system that remembers each user separately.

![](https://i.imgur.com/nJn1o09.png)

## Learning Objectives
In this notebook, you will learn:
1. **Checkpointers** - how LangGraph's persistence layer saves graph state after every step
2. **Threads** - how one `thread_id` keeps one user's conversation separate from everyone else's
3. **In-memory persistence** - `InMemorySaver` for fast, transient experimentation
4. **State history** - replaying the checkpoints a conversation left behind
5. **On-disk persistence** - `SqliteSaver` so conversations survive a restart, and the checkpoint rows it writes

## Prerequisites
- Familiarity with LangGraph basics: `StateGraph`, nodes, edges, `START` / `END`
- Module 01's `03_runnable_with_message_history.ipynb` and `04_multi_user_sessions_and_persistence.ipynb` —
  this notebook is the LangGraph way to do the same thing
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`

### Key Concepts:
- **Checkpointer**: the storage backend that snapshots graph state after each node runs
- **Thread**: a named sequence of checkpoints, addressed by `{"configurable": {"thread_id": "..."}}`
- **Checkpoint**: one immutable snapshot of state — what makes replay and time-travel possible

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and `TAVILY_API_KEY` from the project-root `.env`; the next cell creates
the chat model through the course's shared `aied_helpers` factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the agent's reasoning and tool calls
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧠 Part 1: Defining Graph State

The [State](https://langchain-ai.github.io/langgraph/concepts/low_level/#state) schema is
the input and output contract for every node and edge in the graph.

We use a `TypedDict` with one channel, `messages`. The `add_messages` annotation is a
**reducer**: instead of overwriting the list, it appends to it. That is what lets each node
return only its *new* messages while the conversation accumulates.

In [ ]:
# ============================================================================
# STATE SCHEMA: Conversation Messages
# ============================================================================
# Annotated[list, add_messages] makes `messages` append-only rather than
# overwrite-on-write, so each node returns just the messages it produced.
from typing import Annotated

from langgraph.graph.message import add_messages
from typing_extensions import TypedDict


class State(TypedDict):
    messages: Annotated[list, add_messages]


print("✅ State schema defined!")

### 1.1 🔍 Augmenting the LLM with a Web Search Tool

An agent needs a way to reach information it was not trained on. We wrap Tavily's search
API (`langchain_tavily.TavilySearch`) in a `@tool`-decorated function that returns the top results'
titles, URLs and snippets, and bind it to the LLM.

> **Note**: The tool's **docstring** is what the model reads to decide when to call it.
> Treat it as a prompt, not as developer documentation.

In [ ]:
# ============================================================================
# WEB SEARCH TOOL: Tavily
# ============================================================================
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(max_results=5, search_depth="advanced")


@tool
def search_web(query: str) -> list[dict]:
    """Search the web for a query. Useful for general information or recent news."""
    results = tavily_search.invoke({"query": query}).get("results", [])
    # Titles, URLs and snippets keep the tool output (and the next prompt) small
    return [{"title": r["title"], "url": r["url"], "content": r["content"]} for r in results]


# --- Bind the tool so the model can emit structured tool calls ---
tools = [search_web]
llm_with_tools = llm.bind_tools(tools=tools)

print(f"🔧 Tool bound to LLM: {[t.name for t in tools]}")

---
## 💾 Part 2: In-Memory Persistence

Now we build the agent graph and attach a **checkpointer**. `InMemorySaver` keeps checkpoints
in RAM — perfect for experimentation, gone when the kernel restarts.

The graph itself is the standard tool-calling loop:

```
START → tool_calling_llm → (tools_condition) → tools → tool_calling_llm → ... → END
```

`tools_condition` inspects the model's last message: if it contains tool calls, route to
`tools`; otherwise finish.

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Tool-Calling Agent with In-Memory Checkpointing
# ============================================================================
from datetime import date

from langchain_core.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

# Questions about "latest" news need today's date, or the model searches for stale periods
SYSTEM_PROMPT = SystemMessage(content=f"You are a helpful assistant. Today's date is {date.today():%B %d, %Y}.")


def tool_calling_llm(state: State) -> State:
    """Call the tool-augmented LLM with the system prompt and the conversation so far."""
    response = llm_with_tools.invoke([SYSTEM_PROMPT] + state["messages"])
    return {"messages": [response]}


# --- Nodes ---
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools=tools))

# --- Edges ---
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # tools_condition routes to "tools" when the last message contains tool
    # calls, and to END when the model answered without needing a tool.
    tools_condition,
    ["tools", END],
)
builder.add_edge("tools", "tool_calling_llm")   # the feedback loop

# --- Compile with transient, in-RAM persistence ---
memory = InMemorySaver()
agent_inmem = builder.compile(checkpointer=memory)

print("✅ Agent compiled with in-memory checkpointer!")

In [ ]:
# ============================================================================
# GRAPH STRUCTURE: Nodes and Edges
# ============================================================================
# Printed as text: rendering an image would call an external diagram service.
graph_view = agent_inmem.get_graph()
print("Nodes:", [node_id for node_id in graph_view.nodes])
for edge in graph_view.edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

### 2.1 🧵 Threads: One Conversation Per User

A **thread** is just a name you pass at invocation time:

```python
config = {"configurable": {"thread_id": "user001"}}
```

Every checkpoint the run produces is filed under that id. Pass the same id again and the
agent resumes exactly where it left off; pass a different one and it starts fresh with no
knowledge of the other conversation. That single string is the whole multi-user story.

Below we define two small helpers so each demo turn stays a one-liner.

In [ ]:
# ============================================================================
# DEMO HELPERS: Run a Turn, Inspect a Thread
# ============================================================================

def run_turn(agent, user_input: str, config: dict) -> None:
    """Stream one conversational turn, printing each message and its token usage."""
    for event in agent.stream(
        # A bare string is enough: the add_messages reducer turns it into a HumanMessage
        input={"messages": user_input},
        config=config,
        stream_mode="values",
    ):
        latest = event["messages"][-1]
        latest.pretty_print()
        if getattr(latest, "usage_metadata", None):
            print(f"📋 Token Usage: {latest.usage_metadata}")


def show_thread_state(agent, config: dict) -> None:
    """Print every message currently stored on this thread."""
    current_state = agent.get_state(config)
    for message in current_state.values["messages"]:
        message.pretty_print()
        if getattr(message, "usage_metadata", None):
            print(f"📋 Token Usage: {message.usage_metadata}")


def thread(user_session_id: str) -> dict:
    """Build the config dict that addresses one user's conversation."""
    return {"configurable": {"thread_id": user_session_id}}


print("✅ Demo helpers ready!")

#### 👤 User 001 — First Turn

A simple question that needs no tool. Watch the agent answer directly.

In [ ]:
# ============================================================================
# USER 001: Turn 1
# ============================================================================

config_user001 = thread("user001")

run_turn(agent_inmem, "Explain AI in 1 line", config_user001)

In [ ]:
# ============================================================================
# USER 001: Inspect Stored State
# ============================================================================
# Everything the checkpointer has saved for this thread so far.

show_thread_state(agent_inmem, config_user001)

#### 👤 User 001 — Follow-Up (This Is the Memory Test)

Notice the prompt says *"Do the same for ML"* — it never restates what "the same" means.
The agent only resolves it because the previous turn is replayed from the checkpointer.

In [ ]:
# ============================================================================
# USER 001: Turn 2 - Depends on Turn 1
# ============================================================================

run_turn(agent_inmem, "Do the same for ML", config_user001)

In [ ]:
# ============================================================================
# USER 001: Inspect State Again - Note How It Has Grown
# ============================================================================

show_thread_state(agent_inmem, config_user001)

#### 🕰️ Checkpoint History

The checkpointer keeps *every* snapshot, not just the latest. `get_state_history()` walks
them newest-first — this is the foundation for time-travel and debugging: you can inspect,
fork, or resume from any point.

In [ ]:
# ============================================================================
# CHECKPOINT HISTORY: Every Snapshot on This Thread
# ============================================================================

history = list(agent_inmem.get_state_history(config_user001))

print(f"🕰️ Checkpoints stored for user001: {len(history)}\n")
for snapshot in history:
    step = snapshot.metadata.get("step")
    n_messages = len(snapshot.values.get("messages", []))
    print(f"  step={step:>3}  messages={n_messages:>3}  next={snapshot.next}")

#### 👥 A Second User on a Separate Thread

Same compiled agent, different `thread_id`. This conversation knows nothing about user001's.

In [ ]:
# ============================================================================
# USER 002: A Completely Independent Conversation
# ============================================================================

config_user002 = thread("user002")

run_turn(agent_inmem, "Tell me 3 latest OpenAI product releases", config_user002)

In [ ]:
# ============================================================================
# USER 002: Follow-Up on Their Own Thread
# ============================================================================

run_turn(agent_inmem, "do the same for Meta releases", config_user002)

#### ↩️ Back to User 001 — Their Memory Is Untouched

Switching back to the first thread proves the two conversations never mixed.

In [ ]:
# ============================================================================
# USER 001: Recall - Proves Thread Isolation
# ============================================================================

run_turn(agent_inmem, "what did we discuss so far", config_user001)

---
## 🗄️ Part 3: On-Disk Persistence with SQLite

`InMemorySaver` vanishes with the kernel. `SqliteSaver` writes checkpoints to a file, so a
conversation survives restarts — the minimum bar for anything resembling production.

The API is identical; only the checkpointer changes. Note that `SqliteSaver.from_conn_string()`
is a **context manager**, so the graph is compiled inside the `with` block.

> **Note**: For a real deployment, use `langgraph-checkpoint-postgres` instead — SQLite does
> not handle concurrent writers well.

In [ ]:
# ============================================================================
# SQLITE PERSISTENCE: Checkpoints That Survive a Restart
# ============================================================================
import os

from langgraph.checkpoint.sqlite import SqliteSaver

DB_FILE = "memory.db"
for suffix in ("", "-wal", "-shm"):  # start this demo from an empty database
    if os.path.exists(DB_FILE + suffix):
        os.remove(DB_FILE + suffix)


def call_conversational_agent(agent_graph, prompt: str, user_session_id: str) -> None:
    """Compile the graph against an on-disk checkpointer and run one turn."""
    with SqliteSaver.from_conn_string(DB_FILE) as sqlite_memory:
        agent_extmem = agent_graph.compile(checkpointer=sqlite_memory)
        for event in agent_extmem.stream(
            input={"messages": prompt},
            config=thread(user_session_id),
            stream_mode="values",
        ):
            event["messages"][-1].pretty_print()


print(f"✅ On-disk agent runner ready — state persists to {DB_FILE}")

In [ ]:
# ============================================================================
# ON-DISK DEMO: Turn 1
# ============================================================================

call_conversational_agent(
    agent_graph=builder,
    prompt="What is the latest news on Apple? summarize in 3 bullets",
    user_session_id="bond007",
)

In [ ]:
# ============================================================================
# ON-DISK DEMO: Turn 2 - Recalled from memory.db, Not from RAM
# ============================================================================
# "What about microsoft?" only makes sense if the previous turn was reloaded
# from disk - the graph was recompiled from scratch between the two calls.

call_conversational_agent(
    agent_graph=builder,
    prompt="What about microsoft?",
    user_session_id="bond007",
)

### 3.1 🔎 What the Checkpointer Wrote to Disk

`SqliteSaver` is not a metaphor: `memory.db` is an ordinary SQLite file. Its `checkpoints` table holds one row
per saved snapshot, one for each step the graph took — about five per turn here (the input, the start of the
run, two model calls and a tool call). Its `writes` table holds what each node wrote. Query them directly, then
open a brand-new `SqliteSaver` on the same file, as a restarted process would, and ask `get_state()` for the
thread: the whole conversation comes back from disk.

In [ ]:
# ============================================================================
# INSPECT memory.db: The Rows SqliteSaver Wrote
# ============================================================================
import sqlite3

# Read-only connection: we only look at the checkpointer's tables, never change them
conn = sqlite3.connect(f"file:{DB_FILE}?mode=ro", uri=True)
tables = [name for (name,) in conn.execute("SELECT name FROM sqlite_master WHERE type = 'table'")]
rows = conn.execute("SELECT thread_id, COUNT(*) FROM checkpoints GROUP BY thread_id").fetchall()
conn.close()

print(f"🗄️ Tables in {DB_FILE}: {tables}")
for thread_id, count in rows:
    print(f"   {thread_id}: {count} checkpoint row(s)")

# A fresh SqliteSaver - as after a restart - rebuilds the thread's state from those rows
with SqliteSaver.from_conn_string(DB_FILE) as sqlite_memory:
    restored = builder.compile(checkpointer=sqlite_memory).get_state(thread("bond007"))

messages = restored.values["messages"]
print(f"\n♻️ bond007 restored from disk: {len(messages)} messages")
for m in messages:
    # An AI message that only requests a tool has empty content: show the tool it called instead
    text = str(m.content)[:70] or f"(calls {m.tool_calls[0]['name']})"
    print(f"   {m.type:<6} {text!r}")


---
## 📝 Summary

In this notebook, we learned:

### 1. Checkpointers Give a Graph Memory
- **A checkpointer snapshots state after every node**, not just at the end of a run
- **`InMemorySaver`**: in RAM, fast, disappears with the kernel — use it while experimenting
- **`SqliteSaver`**: on disk, survives restarts — the API is otherwise identical
- Swapping one for the other changes a single line; nothing else in the graph moves
- **`memory.db` is plain SQLite**: one `checkpoints` row per saved step; a fresh `SqliteSaver` rebuilds the
  thread from those rows with `get_state()`

### 2. Threads Separate Users
- **`{"configurable": {"thread_id": "..."}}`** is the entire multi-user mechanism
- The same compiled agent serves every user; the id decides which history is replayed
- We proved isolation by interleaving `user001` and `user002` and returning to the first

### 3. Memory Is Replay, Not Magic
- The agent resolves *"Do the same for ML"* only because prior messages are re-sent
- That means **history costs tokens on every turn** — it grows linearly with the conversation
- `add_messages` is the reducer that accumulates it, so nodes return only new messages

### 4. History Enables Time Travel
- **`get_state_history()`** returns every checkpoint, newest first
- Each snapshot carries `values`, `next`, and `metadata` — enough to inspect, fork, or resume

### 5. Gotchas
- **"Short-term" describes scope, not lifetime.** A thread lives as long as its checkpointer keeps it:
  `SqliteSaver` and `PostgresSaver` keep it on disk until you delete it; only `InMemorySaver` dies with the process
- **`thread_id` must come from trusted server-side config, never from user text.** A user who can pick
  someone else's `thread_id` reads that person's conversation

### Next Steps
- **`02_memory_optimizations.ipynb`** — the direct sequel: because full history grows without
  bound, it bounds what the model sees — sliding-window deletion, trim on read and summarization
- **[`01_cross_thread_memory_store.ipynb`](../../m24_long_term_agentic_memory/notebooks/01a_cross_thread_memory_store.ipynb)**
  (module 03) — memory that outlives a single thread